前言：第六篇，给 Agent 造一个"分身"
上一篇，咱们给 Agent 发了张工单：动手之前先列清单，做一项勾一项，清单本身回流进上下文。从此它干活不跑偏了——做到第八步还记得第一步要干嘛。

但有个成本，从第一篇到现在一直没人算过：上下文本身。

Agent 每读一个文件、每跑一条命令，工具结果都会留在 messages[] 里，一轮不落。你让它查一个"这个项目用什么测试框架"，它读了十个文件才弄明白——那这十个文件的全文，就十条不差地躺在上下文里，直到对话结束。

问题弄清之后，这些中间过程就是占着注意力不干活的数据。而且上下文这东西和硬盘不一样：硬盘存进去就完事了，上下文里的每一个 token，后续每一轮都会被模型重新看一遍。它不是仓库，它是你桌面上一直摊着不收的草稿纸。

这一篇就干一件事：给 Agent 造一个分身——一个叫 task 的工具。主 Agent 遇到调查类的活，派一个子 Agent 出去，给它一张白纸（全新的 messages[]），让它去翻文件、跑命令、把事情查清楚。干完之后，只把结论带回来。中间那几十条工具调用，一条都不进主上下文。

读完你会拿到三样东西：

run_subagent 的完整实现
：一个嵌套的 agent 循环，总共三十来行——你会看到它就是主循环的复刻，唯一的区别是消息列表是新的
一张边界决策表：哪些东西隔离（消息）、哪些东西共享（进程、工作区、权限、Hooks）——每一行都有为什么
两个最容易想错的点：subagent 不是沙箱（它能删你的文件）；委派只有一层（子 Agent 没有 task 工具，套不了娃）
门槛不变：会 Python 基础语法、有上一篇跑通的那份带清单的代码。直接开始。

PART 01：案发现场——查一个问题，上下文胖了十倍
先复原现场。我给之前的 Agent 出过一道很日常的题：

这个项目用什么测试框架？测试怎么跑？帮我确认一下，待会儿改完代码我要跑。

这题不难，甚至可以说平淡。它的执行过程也挑不出毛病：

In [ ]:
> glob("**/test*")
> read_file("pyproject.toml")
> read_file("tests/conftest.py")
> read_file("tests/test_agent.py")
> bash("python -m pytest --collect-only -q")

扫一眼有哪些测试文件，看看依赖声明，读一下 conftest 和一个样例测试，最后空跑一次收集。教科书式的调查流程，最后给我的答案也正确："pytest，直接 python -m pytest 跑，conftest 里有 fixtures"。

但你低头看一眼 messages[]，就是另一副画面了。

五个工具结果，全文躺在里面：pyproject.toml 的完整依赖列表、conftest.py 的四十行代码、test_agent.py 的八十行代码、pytest 的收集输出……加起来好几千 token。而这一切换来的结论，就一行字。
![](有无分身的对比.png)

来算笔账。上下文是按 token 计费的注意力不动产。那几千 token 的文件内容，从它们进入 messages[] 的那一刻起，模型后续每一轮生成都要重新读一遍它们——哪怕这之后的对话聊的是改代码、跑测试、修 bug，跟 conftest.py 没有半毛钱关系。

相当于你查完资料，把十页草稿纸整页订进了作业本，然后每写一行作业都把这十页从头翻一遍。

更要命的是这个模式会累积。修一个 bug 读八个文件，追一个调用链读十二个文件，一天下来，Agent 的上下文里 80% 是"已经用完的调查过程"。真正干活的指令和结论，被自己产生的垃圾数据淹着。

直觉上的解法是把读过的内容删掉？不行。messages[] 是对话的连续性本身，砍中间任何一条，API 直接报错——工具调用和结果必须成对出现。这条路上没有"删除"这个选项。

那就只剩一条路：这些中间过程，从一开始就别进主上下文。

怎么做到？派个分身去干。

PART 02：实现——三十行，一个带白纸出去干活的分身
思路说出来你会觉得朴素得过分：既然主 Agent 的本事就是"一个循环 + 一组工具"，那子 Agent 也照办一个就是了——同样的循环，同样的工具，唯一区别是消息列表从一张白纸开始。

然后把"派分身"这件事本身，包装成一个普通工具：task。主 Agent 调用它，跟调用 read_file 没有任何区别。

先看工具定义，短得不像话：



In [ ]:
TASK_TOOL = {
    "name": "task",
    "description": "Run a subagent with fresh conversation context and return its final text.",
    "input_schema": {
        "type": "object",
        "properties": {"prompt": {"type": "string", "minLength": 1}},
        "required": ["prompt"],
    },
}

TOOLS = [*BASE_TOOLS, TASK_TOOL]
TOOL_HANDLERS = {**BASE_HANDLERS, "task": run_subagent}